<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-09-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-100.

Características do dataset:

- 60.000 imagens RGB
- 100 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar100

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-100 utilizando `tensorflow.keras.datasets.cifar100.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [ ]:
from pathlib import Path
import sys

import mlflow
import pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedShuffleSplit

ROOT_DIR = Path.cwd().resolve()
if not (ROOT_DIR / "src").exists():
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.cifar100_mlp import (
    benchmark_configurations,
    evaluate,
    first_layer_parameter_count,
    load_data,
    load_test_data,
    run_tracked_experiment,
    train_mlp,
)

SEED = 42
BASE_ACTIVATION = "relu"
BASE_HIDDEN_LAYERS = (128, 64)
BASE_LEARNING_RATE = 0.001
MAX_ITER = 20
BATCH_SIZE = 128
TRAIN_SAMPLE_SIZE = 12000
VAL_SAMPLE_SIZE = 2500
EXPERIMENT_NAME = "atividade_05_cifar100_mlp"
experiment_results = {}


def stratified_subset(X, y, n_samples, seed=42):
    if n_samples is None or n_samples >= len(y):
        return X, y

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        train_size=n_samples,
        random_state=seed,
    )
    selected_idx, _ = next(splitter.split(X, y))
    return X[selected_idx], y[selected_idx]


X_train, X_val, y_train, y_val = load_data(SEED)
X_test, y_test = load_test_data()

X_train_exp, y_train_exp = stratified_subset(X_train, y_train, TRAIN_SAMPLE_SIZE, SEED)
X_val_exp, y_val_exp = stratified_subset(X_val, y_val, VAL_SAMPLE_SIZE, SEED)

q1_overview = pd.DataFrame(
    [
        {"etapa": "treino", "shape": X_train.shape, "amostras": len(y_train)},
        {"etapa": "validacao", "shape": X_val.shape, "amostras": len(y_val)},
        {"etapa": "teste", "shape": X_test.shape, "amostras": len(y_test)},
    ]
)
display(q1_overview)
print("Formato original das imagens:", (32, 32, 3))
print("Quantidade de features apos o flatten:", X_train.shape[1])
print("Subconjunto experimental de treino:", X_train_exp.shape)
print("Subconjunto experimental de validacao:", X_val_exp.shape)

### Respostas

1. O formato original de cada imagem e `(32, 32, 3)`, ou seja, 32 pixels de altura, 32 de largura e 3 canais RGB.
2. Depois do flatten, cada imagem passa a ter `3072` features, pois `32 x 32 x 3 = 3072`.
3. O flatten e necessario porque a MLP do `sklearn` recebe vetores de atributos em formato tabular `(n_amostras, n_features)` e nao tensores 3D de imagem.
4. A normalizacao reduz a escala dos pixels para `[0, 1]`, o que melhora a estabilidade numerica, acelera a convergencia e evita que atributos com valores altos dominem o gradiente.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [ ]:
baseline_model = train_mlp(
    X_train=X_train_exp,
    y_train=y_train_exp,
    activation=BASE_ACTIVATION,
    hidden_layers=BASE_HIDDEN_LAYERS,
    learning_rate=BASE_LEARNING_RATE,
    seed=SEED,
    max_iter=MAX_ITER,
    batch_size=BATCH_SIZE,
)

q2_first_layer_params = first_layer_parameter_count(
    X_train.shape[1],
    BASE_HIDDEN_LAYERS[0],
)

print("Arquitetura base:", BASE_HIDDEN_LAYERS)
print("Parametros na primeira camada:", q2_first_layer_params)
print("Iteracoes executadas:", baseline_model.n_iter_)
print("Loss final observada:", baseline_model.loss_)

### Respostas

1. Considerando a arquitetura base `(128, 64)`, a primeira camada possui $3072 \times 128 + 128 = 393344$ parametros treinaveis.
2. A ReLU aplica `max(0, x)`, introduz nao linearidade e ajuda a reduzir o problema de gradientes muito pequenos em comparacao com funcoes saturantes.
3. MLPs possuem muitos parametros em imagens porque cada neuronio denso se conecta a todos os pixels de entrada, fazendo o numero de pesos crescer rapidamente.

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [ ]:
val_metrics, val_predictions = evaluate(baseline_model, X_val, y_val)
test_metrics, test_predictions = evaluate(baseline_model, X_test, y_test)

q3_results = pd.DataFrame(
    [
        {"split": "validacao", **val_metrics},
        {"split": "teste", **test_metrics},
    ]
)
display(q3_results)

Os resultados devem ser interpretados em conjunto. Em CIFAR-100, uma MLP tende a apresentar desempenho mais modesto do que arquiteturas convolucionais, porque o flatten elimina a estrutura espacial da imagem. Mesmo assim, accuracy, precision, recall e f1-score ponderados mostram se o modelo mantem um comportamento consistente entre as 100 classes.

1. Accuracy representa a proporcao global de predicoes corretas.
2. Precision mede a confiabilidade das classes previstas, enquanto recall mede a cobertura dos exemplos realmente pertencentes a cada classe.
3. O f1-score e importante quando se quer equilibrar precision e recall, principalmente em problemas com muitas classes ou com distribuicoes menos uniformes.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [ ]:
mlflow.set_tracking_uri(f"file:{(ROOT_DIR / 'mlruns').resolve()}")

q4_configs = [
    {
        "activation": "relu",
        "hidden_layers": (64,),
        "learning_rate": 0.001,
        "seed": SEED,
        "max_iter": MAX_ITER,
        "batch_size": BATCH_SIZE,
        "run_name": "q4_relu_64_lr001",
    },
    {
        "activation": "relu",
        "hidden_layers": (128, 64),
        "learning_rate": 0.001,
        "seed": SEED,
        "max_iter": MAX_ITER,
        "batch_size": BATCH_SIZE,
        "run_name": "q4_relu_128_64_lr001",
    },
    {
        "activation": "tanh",
        "hidden_layers": (128, 64),
        "learning_rate": 0.001,
        "seed": SEED,
        "max_iter": MAX_ITER,
        "batch_size": BATCH_SIZE,
        "run_name": "q4_tanh_128_64_lr001",
    },
]

q4_results = benchmark_configurations(
    q4_configs,
    X_train_exp,
    y_train_exp,
    X_val_exp,
    y_val_exp,
    experiment_name=f"{EXPERIMENT_NAME}_q4",
)
experiment_results["q4"] = q4_results

display(q4_results)

best_q4 = q4_results.iloc[0]
stable_q4 = q4_results.sort_values(["final_loss", "training_time"]).iloc[0]
print("Tracking URI:", mlflow.get_tracking_uri())
print("Melhor experimento em Q4:", best_q4.to_dict())
print("Configuracao mais estavel em Q4:", stable_q4.to_dict())
print("Para abrir a UI localmente, execute no terminal:")
print(f"mlflow ui --backend-store-uri {(ROOT_DIR / 'mlruns').resolve()}")

### Respostas

1. O melhor experimento desta etapa e o que aparece com maior `accuracy` na tabela registrada no MLflow.
2. A configuracao mais estavel foi tomada como a que combinou menor `final_loss` com menor `training_time`, evitando ganhos pequenos a custo muito alto.
3. O rastreamento experimental facilita comparacoes reprodutiveis, reduz erro manual e preserva o contexto completo de cada treino.

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [ ]:
q5_configs = [
    {
        "activation": activation,
        "hidden_layers": BASE_HIDDEN_LAYERS,
        "learning_rate": BASE_LEARNING_RATE,
        "seed": SEED,
        "max_iter": MAX_ITER,
        "batch_size": BATCH_SIZE,
        "run_name": f"q5_{activation}",
    }
    for activation in ["logistic", "tanh", "relu"]
]

q5_results = benchmark_configurations(
    q5_configs,
    X_train_exp,
    y_train_exp,
    X_val_exp,
    y_val_exp,
    experiment_name=f"{EXPERIMENT_NAME}_q5",
)
experiment_results["q5"] = q5_results

display(q5_results)

best_q5 = q5_results.iloc[0]
stable_q5 = q5_results.sort_values(["final_loss", "training_time"]).iloc[0]
print("Melhor ativacao por accuracy:", best_q5["activation"])
print("Ativacao mais estavel:", stable_q5["activation"])

### Respostas

1. A melhor convergencia e a ativacao que termina com menor `final_loss` sem sacrificar `accuracy`.
2. A ativacao mais estavel e a que apresenta bom desempenho com menor custo temporal e menor oscilacao de perda.
3. Em geral, `relu` tende a convergir mais rapido, `tanh` pode ser competitiva mas mais lenta, e `logistic` costuma sofrer mais com saturacao.
4. A ReLU e amplamente utilizada porque e simples, barata computacionalmente e preserva gradientes mais uteis em redes profundas.

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [ ]:
q6_architectures = [(32,), (64,), (128, 64), (256, 128)]
q6_configs = [
    {
        "activation": BASE_ACTIVATION,
        "hidden_layers": architecture,
        "learning_rate": BASE_LEARNING_RATE,
        "seed": SEED,
        "max_iter": MAX_ITER,
        "batch_size": BATCH_SIZE,
        "run_name": f"q6_{'_'.join(map(str, architecture))}",
    }
    for architecture in q6_architectures
]

q6_results = benchmark_configurations(
    q6_configs,
    X_train_exp,
    y_train_exp,
    X_val_exp,
    y_val_exp,
    experiment_name=f"{EXPERIMENT_NAME}_q6",
)
experiment_results["q6"] = q6_results

display(q6_results)

best_q6 = q6_results.iloc[0]
most_expensive_q6 = q6_results.sort_values("training_time", ascending=False).iloc[0]
print("Melhor arquitetura por accuracy:", best_q6["hidden_layers"])
print("Maior custo computacional:", most_expensive_q6["hidden_layers"])

### Respostas

1. Redes maiores nao melhoram sempre; isso so vale quando o ganho em `accuracy` e `f1_score` compensa o aumento de custo.
2. O melhor tradeoff e a arquitetura que aparece com boa accuracy e tempo de treino moderado.
3. Ha sinal de overfitting quando a rede cresce, o custo aumenta e a validacao quase nao melhora.
4. A arquitetura com maior `training_time` e a de maior custo computacional.

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [ ]:
q7_learning_rates = [0.1, 0.01, 0.001]
q7_configs = [
    {
        "activation": BASE_ACTIVATION,
        "hidden_layers": BASE_HIDDEN_LAYERS,
        "learning_rate": learning_rate,
        "seed": SEED,
        "max_iter": MAX_ITER,
        "batch_size": BATCH_SIZE,
        "run_name": f"q7_lr_{str(learning_rate).replace('.', '_')}",
    }
    for learning_rate in q7_learning_rates
]

q7_results = benchmark_configurations(
    q7_configs,
    X_train_exp,
    y_train_exp,
    X_val_exp,
    y_val_exp,
    experiment_name=f"{EXPERIMENT_NAME}_q7",
)
experiment_results["q7"] = q7_results

display(q7_results)

best_q7 = q7_results.iloc[0]
most_unstable_q7 = q7_results.sort_values(["final_loss", "accuracy"], ascending=[False, True]).iloc[0]
print("Melhor learning rate por accuracy:", best_q7["learning_rate"])
print("Maior instabilidade observada:", most_unstable_q7["learning_rate"])

### Respostas

1. O melhor learning rate e o que aparece com maior `accuracy` e menor `final_loss` na tabela comparativa.
2. O learning rate mais instavel tende a ser o que termina com perda alta e menor consistencia nas metricas.
3. Quando o learning rate e muito alto, o treinamento pode oscilar, divergir ou estacionar em solucoes ruins.
4. Quando o learning rate e muito baixo, o treinamento fica lento e pode nao convergir bem dentro do numero de iteracoes disponiveis.

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

In [ ]:
q8_summary = pd.concat(
    [
        experiment_results["q5"].assign(estudo="ativacao"),
        experiment_results["q6"].assign(estudo="arquitetura"),
        experiment_results["q7"].assign(estudo="learning_rate"),
    ],
    ignore_index=True,
    sort=False,
)

q8_ranking = q8_summary.sort_values(
    ["accuracy", "f1_score", "final_loss"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(q8_ranking)

best_overall = q8_ranking.iloc[0]
most_stable_overall = q8_ranking.sort_values(["final_loss", "training_time"]).iloc[0]
print("Melhor configuracao geral:", best_overall.to_dict())
print("Configuracao mais estavel:", most_stable_overall.to_dict())

### Discussao

Os experimentos mostram um comportamento tipico de MLPs em problemas visuais. A `loss` tende a cair melhor quando a taxa de aprendizado e moderada e a ativacao nao satura cedo. Learning rates muito altos geralmente tornam o treino instavel, enquanto valores muito baixos tornam a convergencia lenta. Arquiteturas maiores aumentam a capacidade do modelo, mas tambem elevam o custo computacional e nem sempre entregam ganhos proporcionais em validacao. Entre as ativacoes, a ReLU costuma oferecer o melhor equilibrio entre velocidade de treinamento e estabilidade.

As principais limitacoes da MLP neste problema surgem do flatten. Ao transformar a imagem em vetor, a rede perde relacoes espaciais locais, bordas e padroes hierarquicos que arquiteturas convolucionais conseguem explorar melhor. O backpropagation continua sendo essencial, porque ele calcula os gradientes camada a camada e orienta a atualizacao dos pesos para reduzir a funcao de perda, mas ele nao resolve sozinho a limitacao estrutural de uma arquitetura totalmente densa para imagens.

1. A melhor configuracao final e a primeira linha da tabela consolidada em Q8.
2. As principais dificuldades observadas foram o grande numero de classes, o alto numero de parametros e o custo computacional das camadas densas.
3. MLPs possuem limitacoes para imagens porque tratam pixels como atributos independentes e nao aproveitam a estrutura espacial local.
4. O backpropagation contribui para o aprendizado ao propagar o erro da saida para as camadas anteriores, ajustando os pesos na direcao que reduz a loss.